In [0]:
%sql
USE CATALOG `retail-sales`;
USE SCHEMA `03_gold`;

In [0]:
%sql

-- =====================================================
-- 1. DAILY SALES SUMMARY
-- =====================================================

CREATE OR REPLACE TABLE `retail-sales`.03_gold.DailySalesSummary
USING DELTA
AS

SELECT

    TxnDate,

    COUNT(TransactionID) AS TotalTransactions,

    SUM(Quantity) AS TotalQuantity,

    ROUND(SUM(Amount), 2) AS TotalSalesAmount

FROM `retail-sales`.02_silver.FactSales

GROUP BY TxnDate;



In [0]:

-- =====================================================
-- 2. STORE SALES SUMMARY
-- =====================================================

CREATE OR REPLACE TABLE `retail-sales`.03_gold.StoreSalesSummary
USING DELTA
AS

SELECT

    ds.StoreID,

    ds.StoreName,

    ds.Region,

    COUNT(fs.TransactionID) AS TotalTransactions,

    SUM(fs.Quantity) AS TotalQuantitySold,

    ROUND(SUM(fs.Amount), 2) AS TotalSalesAmount

FROM `retail-sales`.02_silver.FactSales fs

INNER JOIN `retail-sales`.02_silver.DimStore ds
    ON fs.StoreSK = ds.StoreSK

GROUP BY
    ds.StoreID,
    ds.StoreName,
    ds.Region;


In [0]:

-- =====================================================
-- 3. PRODUCT CATEGORY SALES SUMMARY
-- =====================================================

CREATE OR REPLACE TABLE `retail-sales`.03_gold.ProductCategorySummary
USING DELTA
AS

SELECT

    dp.Category,

    COUNT(fs.TransactionID) AS TotalTransactions,

    SUM(fs.Quantity) AS TotalQuantitySold,

    ROUND(SUM(fs.Amount), 2) AS TotalSalesAmount,

    ROUND(AVG(fs.Amount), 2) AS AverageTransactionValue

FROM `retail-sales`.02_silver.FactSales fs

INNER JOIN `retail-sales`.02_silver.DimProduct dp
    ON fs.ProductSK = dp.ProductSK

GROUP BY dp.Category;



In [0]:

-- =====================================================
-- 4. CUSTOMER SALES SUMMARY
-- =====================================================

CREATE OR REPLACE TABLE `retail-sales`.03_gold.CustomerSalesSummary
USING DELTA
AS

SELECT

    dc.CustomerID,

    dc.CustomerName,

    dc.City,

    COUNT(fs.TransactionID) AS TotalTransactions,

    SUM(fs.Quantity) AS TotalQuantityPurchased,

    ROUND(SUM(fs.Amount), 2) AS TotalAmountSpent,

    ROUND(AVG(fs.Amount), 2) AS AveragePurchaseValue

FROM `retail-sales`.02_silver.FactSales fs

INNER JOIN `retail-sales`.02_silver.DimCustomer dc
    ON fs.CustomerSK = dc.CustomerSK

WHERE dc.IsActive = 1

GROUP BY
    dc.CustomerID,
    dc.CustomerName,
    dc.City;



In [0]:

-- =====================================================
-- 5. REGION SALES SUMMARY
-- =====================================================

CREATE OR REPLACE TABLE `retail-sales`.03_gold.RegionSalesSummary
USING DELTA
AS

SELECT

    ds.Region,

    COUNT(DISTINCT ds.StoreID) AS TotalStores,

    COUNT(fs.TransactionID) AS TotalTransactions,

    SUM(fs.Quantity) AS TotalQuantitySold,

    ROUND(SUM(fs.Amount), 2) AS TotalSalesAmount

FROM `retail-sales`.02_silver.FactSales fs

INNER JOIN `retail-sales`.02_silver.DimStore ds
    ON fs.StoreSK = ds.StoreSK

GROUP BY ds.Region;

